# Marginals and Conditionals of Gaussians

## Learning Objectives
- Partition a joint Gaussian into blocks and read off the **marginal** directly
- Derive the **conditional Gaussian** formula: $x_1 | x_2 \sim \mathcal{N}(\mu_{1|2}, \Sigma_{1|2})$
- Understand the **Schur complement** $\Sigma_{1|2} = \Sigma_{11} - \Sigma_{12}\Sigma_{22}^{-1}\Sigma_{21}$
- Verify these formulas empirically by sampling from a joint Gaussian
- Recognise these results as the key tools for the Factor Analysis E-step

## Problem Statement

Suppose $x = \begin{bmatrix} x_1 \\ x_2 \end{bmatrix} \sim \mathcal{N}(\mu, \Sigma)$ where $x_1 \in \mathbb{R}^r$, $x_2 \in \mathbb{R}^s$, and:
$$\mu = \begin{bmatrix} \mu_1 \\ \mu_2 \end{bmatrix}, \qquad
\Sigma = \begin{bmatrix} \Sigma_{11} & \Sigma_{12} \\ \Sigma_{21} & \Sigma_{22} \end{bmatrix}$$

**Marginal distribution:**
$$x_1 \sim \mathcal{N}(\mu_1,\; \Sigma_{11})$$

**Conditional distribution:**
$$x_1 \mid x_2 \sim \mathcal{N}\bigl(\mu_{1|2},\; \Sigma_{1|2}\bigr)$$
$$\mu_{1|2} = \mu_1 + \Sigma_{12}\Sigma_{22}^{-1}(x_2 - \mu_2)$$
$$\Sigma_{1|2} = \Sigma_{11} - \Sigma_{12}\Sigma_{22}^{-1}\Sigma_{21}$$

The conditional covariance $\Sigma_{1|2}$ is called the **Schur complement** of $\Sigma_{22}$ in $\Sigma$.
It is always $\preceq \Sigma_{11}$ — conditioning on $x_2$ reduces uncertainty about $x_1$.

These two results are used directly in the Factor Analysis E-step to compute $p(z \mid x; \mu, \Lambda, \Psi)$.

## 1. Partitioned Covariance Structure

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import multivariate_normal

np.random.seed(42)

# 3D joint: x1 (2D) and x2 (1D)
mu_joint = np.array([1.0, 2.0, 3.0])
Sigma_joint = np.array([
    [2.0,  0.8, 1.2],
    [0.8,  1.5, 0.6],
    [1.2,  0.6, 1.0],
])

# Partition: x1 = first 2 dims, x2 = last 1 dim
r, s = 2, 1
mu1, mu2 = mu_joint[:r], mu_joint[r:]
S11 = Sigma_joint[:r, :r]
S12 = Sigma_joint[:r, r:]
S21 = Sigma_joint[r:, :r]
S22 = Sigma_joint[r:, r:]

print('Joint mean μ =', mu_joint)
print('Joint covariance Σ =')
print(Sigma_joint)
print(f'\nPartition: x1 ∈ R^{r}, x2 ∈ R^{s}')
print(f'Σ11 =\n{S11}')
print(f'Σ12 =\n{S12}')
print(f'Σ22 =\n{S22}')

# Marginal
print('\n--- Marginal x1 ~ N(mu1, Sigma11) ---')
print(f'mu1 = {mu1}')
print(f'Sigma11 =\n{S11}')

# Conditional (at x2 = 3.5)
x2_obs = np.array([3.5])
S22_inv = np.linalg.inv(S22)
mu_cond   = mu1 + S12 @ S22_inv @ (x2_obs - mu2)
Sig_cond  = S11 - S12 @ S22_inv @ S21
print(f'\n--- Conditional x1|x2={x2_obs[0]} ~ N(mu_cond, Sig_cond) ---')
print(f'mu_cond = {mu_cond.round(4)}')
print(f'Sig_cond =\n{Sig_cond.round(4)}')
print(f'\nNote: Sig_cond ≤ S11 (conditioning reduces uncertainty)')
print(f'  Trace(S11) = {np.trace(S11):.4f}')
print(f'  Trace(Sig_cond) = {np.trace(Sig_cond):.4f}')

## 2. Empirical Verification by Sampling

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import multivariate_normal

np.random.seed(0)

# 2D joint for easy visualisation
mu_j   = np.array([1.0, 2.0])
Sig_j  = np.array([[2.0, 1.2], [1.2, 1.5]])
n_samp = 5000
samples = np.random.multivariate_normal(mu_j, Sig_j, n_samp)

# Partition: x1 = col 0, x2 = col 1
mu1_j, mu2_j   = mu_j[0], mu_j[1]
S11_j, S12_j   = Sig_j[0, 0], Sig_j[0, 1]
S21_j, S22_j   = Sig_j[1, 0], Sig_j[1, 1]

# Conditional at several x2 values
x2_vals = [0.0, 2.0, 4.0]
tol     = 0.2   # window around x2

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

x_plot = np.linspace(-3, 6, 300)

# Left: joint scatter
ax = axes[0]
ax.scatter(samples[:, 0], samples[:, 1], s=3, alpha=0.2, c='steelblue')
for x2v in x2_vals:
    ax.axhline(x2v, ls='--', lw=1.5, alpha=0.8)
ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
ax.set_title('Joint distribution $p(x_1, x_2)$\n(dashed = conditioning slices)')

colors = ['#2166ac', '#d6604d', '#4dac26']

# Middle: marginal x1 — theoretical vs empirical
ax = axes[1]
ax.hist(samples[:, 0], bins=60, density=True, color='steelblue', alpha=0.5, label='Empirical')
from scipy.stats import norm
ax.plot(x_plot, norm.pdf(x_plot, mu1_j, np.sqrt(S11_j)), 'k-', lw=2.5,
        label=f'Theory: N({mu1_j}, {S11_j})')
ax.set_xlabel('$x_1$'); ax.set_ylabel('Density')
ax.set_title('Marginal $p(x_1) = \\mathcal{N}(\\mu_1, \\Sigma_{11})$')
ax.legend(fontsize=8.5)

# Right: conditional x1 | x2 for three x2 values
ax = axes[2]
for i, x2v in enumerate(x2_vals):
    mask = np.abs(samples[:, 1] - x2v) < tol
    x1_cond = samples[mask, 0]
    mu_c  = mu1_j + S12_j / S22_j * (x2v - mu2_j)
    sig_c = np.sqrt(S11_j - S12_j**2 / S22_j)
    ax.hist(x1_cond, bins=30, density=True, alpha=0.35, color=colors[i])
    ax.plot(x_plot, norm.pdf(x_plot, mu_c, sig_c), color=colors[i], lw=2.5,
            label=f'$x_2={x2v}$: N({mu_c:.2f}, {sig_c**2:.2f})')

ax.set_xlabel('$x_1$'); ax.set_ylabel('Density')
ax.set_title('Conditional $p(x_1|x_2)$\nMean shifts, variance constant')
ax.legend(fontsize=8)

fig.suptitle('Marginal and Conditional Distributions of a Partitioned Gaussian',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print('Theoretical conditional variance: Σ_{1|2} =', S11_j - S12_j**2/S22_j)
print('Note: same for all x2 values — only the mean shifts')

## 3. Conditional Mean and Variance as Functions of $x_2$

The conditional mean is a **linear function** of $x_2$:
$$\mu_{1|2}(x_2) = \mu_1 + \Sigma_{12}\Sigma_{22}^{-1}(x_2 - \mu_2)$$

The conditional covariance $\Sigma_{1|2}$ is **constant** (does not depend on $x_2$).
This is a defining property of Gaussian distributions — linear conditional means.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(1)

# 2D Gaussian with strong correlation
mu_j  = np.array([0.0, 0.0])
rho   = 0.85
Sig_j = np.array([[1.0, rho], [rho, 1.0]])
n_samp = 4000
samples = np.random.multivariate_normal(mu_j, Sig_j, n_samp)

S11, S12, S22 = Sig_j[0, 0], Sig_j[0, 1], Sig_j[1, 1]
Sig_cond = S11 - S12**2 / S22

x2_range = np.linspace(-3, 3, 100)
mu_cond_range = mu_j[0] + S12/S22 * (x2_range - mu_j[1])

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

# Left: scatter + conditional mean line
ax = axes[0]
ax.scatter(samples[:, 1], samples[:, 0], s=5, alpha=0.25, c='steelblue')
ax.plot(x2_range, mu_cond_range, 'r-', lw=2.5,
        label=f'$\\mu_{{1|2}}(x_2) = \\rho x_2$ (linear)')
ax.fill_between(x2_range,
                mu_cond_range - 2*np.sqrt(Sig_cond),
                mu_cond_range + 2*np.sqrt(Sig_cond),
                alpha=0.2, color='red', label=f'±2σ, σ²={Sig_cond:.3f} (constant)')
ax.set_xlabel('$x_2$ (observed)')
ax.set_ylabel('$x_1$ (to predict)')
ax.set_title(f'Conditional mean: linear in $x_2$\n'
             f'Conditional variance: constant = {Sig_cond:.3f} (ρ={rho})')
ax.legend(fontsize=9)

# Right: effect of correlation strength on Sig_cond
ax = axes[1]
rho_vals = np.linspace(-0.999, 0.999, 200)
sig_cond_vals = 1 - rho_vals**2   # S11=1, S22=1, S12=rho
ax.plot(rho_vals, sig_cond_vals, 'b-', lw=2.5)
ax.axhline(1.0, color='gray', ls='--', lw=1.5, label='$\\Sigma_{11}$ (unconditional)')
ax.axhline(0.0, color='k',    ls='--', lw=1.5, alpha=0.5)
ax.scatter([rho], [Sig_cond], s=150, c='red', zorder=6,
           label=f'Current ρ={rho}, Σ_{{1|2}}={Sig_cond:.3f}')
ax.set_xlabel('Correlation $\\rho = \\Sigma_{12}$')
ax.set_ylabel('Conditional variance $\\Sigma_{1|2}$')
ax.set_title('Conditioning reduces uncertainty\n'
             '$\\Sigma_{1|2} = \\Sigma_{11} - \\Sigma_{12}^2/\\Sigma_{22} \\leq \\Sigma_{11}$')
ax.legend(fontsize=9)

fig.suptitle('Conditional Gaussian: Linear Mean, Constant Variance (Schur Complement)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Extension to Block Matrices and Factor Analysis Preview

In Factor Analysis the joint distribution of $(z, x)$ is Gaussian with block structure:
$$\begin{bmatrix} z \\ x \end{bmatrix} \sim \mathcal{N}\!\left(
\begin{bmatrix} 0 \\ \mu \end{bmatrix},
\begin{bmatrix} I & \Lambda^T \\ \Lambda & \Lambda\Lambda^T + \Psi \end{bmatrix}\right)$$

Applying the conditional formula directly gives the **E-step posterior**:
$$z \mid x \sim \mathcal{N}\bigl(\underbrace{\Lambda^T(\Lambda\Lambda^T + \Psi)^{-1}(x - \mu)}_{\mu_{z|x}},\.
\underbrace{I - \Lambda^T(\Lambda\Lambda^T + \Psi)^{-1}\Lambda}_{\Sigma_{z|x}}\bigr)$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import multivariate_normal

np.random.seed(7)

# Small Factor Analysis example: n=3 observed, k=2 latent
n, k = 3, 2
mu_fa = np.array([1.0, 0.5, -0.5])
Lambda = np.array([[1.0, 0.3],
                   [0.5, 1.0],
                   [0.2, 0.8]])
Psi = np.diag([0.3, 0.4, 0.2])

# Marginal covariance of x
Sig_x = Lambda @ Lambda.T + Psi

# E-step posterior formula
M = Lambda.T @ np.linalg.inv(Sig_x)
Sig_zx = np.eye(k) - M @ Lambda   # Posterior covariance (constant in x)

print('Factor Analysis Setup:')
print(f'  n (observed dims) = {n},  k (latent dims) = {k}')
print(f'  Λ (loading matrix) =\n{Lambda}')
print(f'  Ψ (noise covar, diagonal) =\n{Psi}')
print(f'  Σ_x = ΛΛᵀ + Ψ =\n{Sig_x.round(3)}')

print(f'\nE-step posterior covariance Σ_{{z|x}} (same for all x):')
print(Sig_zx.round(4))

# Sample some x points and compute posteriors
m_demo = 5
X_demo = np.random.multivariate_normal(mu_fa, Sig_x, m_demo)

print('\nSample posterior means μ_{z|x^(i)} for 5 points:')
for i, xi in enumerate(X_demo):
    mu_zi = M @ (xi - mu_fa)
    print(f'  x^({i+1}) = {xi.round(2)}  →  μ_{{z|x}} = {mu_zi.round(3)}')

# Visualise: joint (z,x) for k=n=1 case
n1, k1 = 1, 1
lam_s, psi_s, mu_s = 1.5, 0.5, 0.0
Sig_x1 = lam_s**2 + psi_s
Sig_zx1 = 1 - lam_s**2 / Sig_x1

Sig_joint1 = np.array([[1.0, lam_s], [lam_s, Sig_x1]])
mu_joint1  = np.array([0.0, mu_s])
samples1 = np.random.multivariate_normal(mu_joint1, Sig_joint1, 2000)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
ax.scatter(samples1[:, 0], samples1[:, 1], s=6, alpha=0.3, c='steelblue')

z_range  = np.linspace(-3, 3, 100)
mu_x_given_z = mu_s + lam_s * z_range
ax.plot(z_range, mu_x_given_z, 'r-', lw=2.5, label=f'E[x|z] = μ + Λz = {lam_s}z')
ax.set_xlabel('$z$ (latent)'); ax.set_ylabel('$x$ (observed)')
ax.set_title('Joint $(z, x)$ in Factor Analysis\n$x = \\mu + \\Lambda z + \\epsilon$')
ax.legend(fontsize=9)

ax = axes[1]
x_range = np.linspace(-5, 5, 100)
mu_z_given_x = lam_s / Sig_x1 * (x_range - mu_s)   # posterior mean
ax.scatter(samples1[:, 1], samples1[:, 0], s=6, alpha=0.3, c='steelblue')
ax.plot(x_range, mu_z_given_x, 'g-', lw=2.5,
        label=f'E[z|x] = Λᵀ(ΛΛᵀ+Ψ)⁻¹(x−μ)')
ax.fill_between(x_range,
                mu_z_given_x - 2*np.sqrt(Sig_zx1),
                mu_z_given_x + 2*np.sqrt(Sig_zx1),
                alpha=0.2, color='green', label=f'±2σ, Σ_{{z|x}}={Sig_zx1:.3f}')
ax.set_xlabel('$x$ (observed)'); ax.set_ylabel('$z$ (latent)')
ax.set_title('Posterior $p(z|x)$ — the E-step\nLinear mean, constant variance')
ax.legend(fontsize=9)

fig.suptitle('Conditional Gaussians in Factor Analysis: $z|x$ Posterior',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Derivation Pathway

### Derivation pathway

In [ ]:
from IPython.display import HTML
HTML("""
<svg xmlns="http://www.w3.org/2000/svg" width="780" height="468"
     viewBox="0 0 780 468" font-family="Segoe UI, Arial, sans-serif">
  <defs>
    <marker id="ah" markerWidth="10" markerHeight="7" refX="9" refY="3.5"
            orient="auto" markerUnits="userSpaceOnUse">
      <polygon points="0 0,10 3.5,0 7" fill="#444"/>
    </marker>
    <marker id="ahd" markerWidth="10" markerHeight="7" refX="9" refY="3.5"
            orient="auto" markerUnits="userSpaceOnUse">
      <polygon points="0 0,10 3.5,0 7" fill="#999"/>
    </marker>
  </defs>
  <rect x="10" y="12" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="35" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">Joint Gaussian</text>
  <text x="102" y="52" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">x ~ N(&#x3bc;, &#x3a3;)</text>
  <line x1="197" y1="35" x2="216" y2="35"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="12" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="40" font-size="13" text-anchor="middle" fill="#333"
        >x = [x&#x2081;; x&#x2082;] with partitioned &#x3bc; and &#x3a3;</text>
  <line x1="102" y1="58" x2="102" y2="120"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>
  <text x="114" y="80" font-size="11.5" font-style="italic" fill="#555"
        >partition &#x3a3; into &#x3a3;&#x2081;&#x2081;, &#x3a3;&#x2081;&#x2082;, &#x3a3;&#x2082;&#x2082;</text>
  <rect x="10" y="112" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="140" font-size="13.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">Marginals</text>
  <line x1="197" y1="135" x2="216" y2="135"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="112" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="140" font-size="13" text-anchor="middle" fill="#333"
        >x&#x2081; ~ N(&#x3bc;&#x2081;, &#x3a3;&#x2081;&#x2081;) and x&#x2082; ~ N(&#x3bc;&#x2082;, &#x3a3;&#x2082;&#x2082;)</text>
  <line x1="102" y1="158" x2="102" y2="220"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>
  <text x="114" y="180" font-size="11.5" font-style="italic" fill="#555"
        >derive conditional using Schur complement</text>
  <rect x="10" y="212" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="235" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">Conditional</text>
  <text x="102" y="252" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">x&#x2081;|x&#x2082;</text>
  <line x1="197" y1="235" x2="216" y2="235"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="212" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="240" font-size="13" text-anchor="middle" fill="#333"
        >mean: &#x3bc;&#x2081; + &#x3a3;&#x2081;&#x2082;&#x3a3;&#x2082;&#x2082;&#x207b;&#xb9;(x&#x2082;-&#x3bc;&#x2082;); cov: &#x3a3;&#x2081;&#x2081; &#x2212; &#x3a3;&#x2081;&#x2082;&#x3a3;&#x2082;&#x2082;&#x207b;&#xb9;&#x3a3;&#x2082;&#x2081;</text>
  <line x1="102" y1="258" x2="102" y2="320"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>
  <text x="114" y="280" font-size="11.5" font-style="italic" fill="#555"
        >still Gaussian &#x2014; linear function of x&#x2082;</text>
  <rect x="10" y="312" width="185" height="46" rx="7"
        fill="#1a5fa8" stroke="#1a5fa8" stroke-width="2"/>
  <text x="102" y="335" font-size="12.5" font-weight="700"
        text-anchor="middle" fill="#ffffff">Conditional</text>
  <text x="102" y="352" font-size="12.5" font-weight="700"
        text-anchor="middle" fill="#ffffff">is Gaussian</text>
  <line x1="197" y1="335" x2="216" y2="335"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="312" width="548" height="46" rx="7"
        fill="#dce8f8" stroke="#7aadd4" stroke-width="1.5"/>
  <text x="495" y="340" font-size="13" text-anchor="middle" fill="#333"
        >exact formula &#x2014; foundation for Kalman filter and factor analysis</text>
</svg>
""")

## Summary

| Concept | Formula | Key Insight |
|---|---|---|
| Joint Gaussian | $x = (x_1, x_2)^T \sim \mathcal{N}(\mu, \Sigma)$ | Partition mean and covariance by block |
| Marginals | $x_1 \sim \mathcal{N}(\mu_1, \Sigma_{11})$ | Marginalising a joint Gaussian gives Gaussian |
| Conditional mean | $\mu_{1|2} = \mu_1 + \Sigma_{12}\Sigma_{22}^{-1}(x_2 - \mu_2)$ | Linear function of observed $x_2$ |
| Conditional covariance | $\Sigma_{1|2} = \Sigma_{11} - \Sigma_{12}\Sigma_{22}^{-1}\Sigma_{21}$ | Schur complement — reduced by observing $x_2$ |
| Conditional density | $x_1 \lvert x_2 \sim \mathcal{N}(\mu_{1|2}, \Sigma_{1|2})$ | Still Gaussian — Gaussians are closed under conditioning |

**Key insight:** the conditional distribution of a jointly Gaussian vector is Gaussian with mean linear in the observed component and covariance reduced by the Schur complement — this formula is the core of Kalman filtering and the E-step in factor analysis.